# Results, ablation study and comparison — Cost-Aware Multi-Agent RAG for Hard Drive Failure Prediction

This notebook shows the evaluation of the project with tables and graphs. It needs **no GPU** and no LLM:
it only reads the saved results.

**How to run in Colab**
1. Upload `results_bundle.zip` (made by `python -m scripts.export_results` in `server/`) to
   `MyDrive/drive-failure-agents/`, or upload it when the first cell asks.
2. `Runtime → Run all` (a CPU runtime is enough; it takes about a minute).

Every number below is computed from the bundle; nothing is typed in by hand.

| Section | What it shows |
|---|---|
| 1 | Dataset and splits |
| 2 | Main results (3 seeds) |
| 3 | Accuracy, precision, recall, F1, confusion matrix — and why accuracy alone misleads |
| 4 | ROC, precision–recall, calibration, cost and selective-prediction graphs |
| 5 | **Ablation study** (remove one component at a time) |
| 6 | **LLM ablation** (live graph runs: planner, retrieval, explainer) |
| 7 | **Comparison with existing work** |
| 8 | Conclusions |

In [ ]:
import json, os, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from IPython.display import Markdown, display
from sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score,
                             confusion_matrix, f1_score, matthews_corrcoef, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score, roc_curve)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

BUNDLE = "results_bundle.zip"
candidates = [Path(os.environ.get("RESULTS_BUNDLE", BUNDLE)),
              Path("/content/drive/MyDrive/drive-failure-agents") / BUNDLE,
              Path("server/outputs") / BUNDLE]
try:
    from google.colab import drive, files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB and not any(p.exists() for p in candidates):
    try:
        drive.mount("/content/drive")
    except Exception as exc:
        print("Google Drive not mounted:", exc)
bundle = next((p for p in candidates if p.exists()), None)
if bundle is None and IN_COLAB:
    print(f"{BUNDLE} not found on Drive; please upload it now.")
    uploaded = files.upload()
    bundle = Path(next(iter(uploaded)))
if bundle is None:
    raise FileNotFoundError(f"{BUNDLE} not found; looked in {[str(p) for p in candidates]}")

ROOT = Path("results_bundle")
with zipfile.ZipFile(bundle) as zf:
    zf.extractall(ROOT)
FINAL, LLM = ROOT / "final", ROOT / "llm"
summary = json.loads((ROOT / "dataset_summary.json").read_text(encoding="utf-8"))
metrics = json.loads((FINAL / "metrics.json").read_text(encoding="utf-8"))
config = yaml.safe_load((FINAL / "config.yaml").read_text(encoding="utf-8"))["resolved"]
agg = pd.DataFrame(metrics["aggregate"])
preds = {s: pd.read_parquet(FINAL / "predictions" / f"{s}.parquet")
         for s in ("test", "unseen") if (FINAL / "predictions" / f"{s}.parquet").exists()}
H = max(summary["horizons"])
R = config["decision"]["default_cost_ratio"]
SEEDS = metrics["seeds"]

NAMES = {
    "full_system": "Full system (ours)",
    "multi_agent_no_rag": "Agents without retrieval",
    "sequence": "1D-CNN sequence model",
    "lgbm_global": "Single LightGBM, all features",
    "single_llm_agent": "Single LLM agent",
    "majority_vote": "Majority voting",
    "no_calibration": "No calibration",
    "no_media_health": "– media-health agent",
    "no_mechanical": "– mechanical agent",
    "no_thermal": "– thermal agent",
    "no_usage_wear": "– usage/wear agent",
    "no_case_retrieval": "– case retrieval (RAG)",
    "no_knowledge_retrieval": "– knowledge retrieval (RAG)",
    "no_cost_policy": "– cost policy (fixed threshold)",
    "no_drift_recalibration": "– drift recalibration",
    "no_planner": "– LLM planner",
}
KEY = ["full_system", "multi_agent_no_rag", "sequence", "lgbm_global", "majority_vote",
       "no_calibration"]
name = lambda s: NAMES.get(s, s)

print(f"bundle: {bundle}  |  experiment: {metrics['id']}  |  seeds: {SEEDS}  |  max horizon: {H} days")
print("prediction sets:", {s: len(d) for s, d in preds.items()})

## 1. Dataset and splits

Backblaze Drive Stats: one row per drive per day with SMART counters and a failure flag.
The four drive models with the most failures are used for training; a fifth model is held out completely
(**unseen**) to test generalisation to a drive model the system never saw.

In [ ]:
per_model = pd.DataFrame(summary["per_model"])
held = set(summary["models"].get("holdout_models", []))
per_model["role"] = np.where(per_model["model"].isin(held), "unseen (held out)", "training")
per_model["failure rate / drive-year (%)"] = (100 * per_model["failures"]
                                             / (per_model["drive_days"] / 365)).round(2)
display(per_model[["model", "role", "drives", "drive_days", "failures",
                   "failure rate / drive-year (%)", "first_date", "last_date"]])
tot = per_model[["drives", "drive_days", "failures"]].sum()
print(f"Total: {int(tot.drives):,} drives, {int(tot.drive_days):,} drive-days, "
      f"{int(tot.failures):,} failed drives")

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
colors = ["#94a3b8" if r.startswith("unseen") else "#2563eb" for r in per_model["role"]]
ax[0].barh(per_model["model"], per_model["drives"], color=colors)
ax[0].set_title("Drives per model (grey = unseen)")
ax[1].barh(per_model["model"], per_model["failures"], color=colors)
ax[1].set_title("Failed drives per model")
plt.tight_layout(); plt.show()

win = pd.DataFrame(summary["windows"]).T
for h in summary["horizons"]:
    win[f"positive rate {h}d (%)"] = (100 * win[f"positives_{h}"] / win["windows"]).round(3)
display(Markdown("**Evaluation windows** (a window = the last 30 days of one drive; label = fails within *h* days). "
                 "Training is balanced by keeping all failures plus sampled healthy windows; "
                 "validation, test and unseen keep the real failure rate."))
display(win)
p = summary["splits"].get("periods", {})
display(Markdown(f"**Split by drive *and* time:** train up to {p.get('train_end')}, validation to "
                 f"{p.get('val_end')}, test after that (to {p.get('data_end')}). "
                 "No drive appears in two sets, and no set sees data from a later period."))

## 2. Main results (mean ± std over three seeds)

* **AUPRC** — area under the precision–recall curve; the main metric for rare events (random guessing ≈ the positive rate).
* **ROC-AUC** — chance that a failing window is ranked above a healthy one.
* **Recall @1% FAR** — share of failing windows caught when only 1% of healthy windows raise a false alarm.
* **ECE** — calibration error (0 = predicted probabilities match observed rates).
* **Detected / lead days** — share of failing drives flagged before failure, and how many days ahead.
* **Cost** — realised maintenance cost per window at cost ratio *R* (1 = one planned replacement).

In [ ]:
def ms(row, col, pct=False, nd=3):
    m, s = row.get(f"{col}_mean"), row.get(f"{col}_std")
    if m is None or pd.isna(m):
        return "–"
    k = 100 if pct else 1
    return f"{k * m:.{nd}f} ± {k * s:.{nd}f}" if s is not None and not pd.isna(s) else f"{k * m:.{nd}f}"

def results_table(set_name, systems):
    a = agg[agg["set"] == set_name].set_index("system")
    rows = []
    for s in systems:
        if s not in a.index:
            continue
        r = a.loc[s]
        rows.append({"system": name(s),
                     **{f"AUPRC {h}d": ms(r, f"auprc_{h}") for h in summary["horizons"]},
                     f"ROC-AUC {H}d": ms(r, f"roc_auc_{H}"),
                     "Recall @1% FAR": ms(r, f"recall@far0.01_{H}"),
                     f"ECE {H}d": ms(r, f"ece_{H}", nd=4),
                     "Detected (%)": ms(r, "lead_detected_share", pct=True, nd=1),
                     "Lead (days)": ms(r, "lead_mean_lead_days", nd=1),
                     f"Cost R={R:g}": ms(r, f"cost_model_r{R:g}")})
    return pd.DataFrame(rows)

for set_name in ("test", "unseen"):
    display(Markdown(f"**{set_name.capitalize()} set** "
                     + ("(later time period, training drive models)" if set_name == "test"
                        else "(drive model never seen in training)")))
    display(results_table(set_name, KEY))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, set_name in zip(axes, ("test", "unseen")):
    a = agg[agg["set"] == set_name].set_index("system")
    hs = summary["horizons"]
    width = 0.8 / len(KEY)
    for i, s in enumerate([k for k in KEY if k in a.index]):
        means = [a.loc[s, f"auprc_{h}_mean"] for h in hs]
        stds = [a.loc[s, f"auprc_{h}_std"] for h in hs]
        ax.bar(np.arange(len(hs)) + i * width, means, width, yerr=stds, capsize=2, label=name(s))
    ax.set_xticks(np.arange(len(hs)) + 0.4 - width / 2, [f"{h} days" for h in hs])
    ax.set_title(f"AUPRC by horizon ({set_name}, mean ± std over seeds)")
axes[0].set_ylabel("AUPRC")
axes[1].legend(fontsize=8, loc="upper left")
plt.tight_layout(); plt.show()

## 3. Accuracy-style metrics and the confusion matrix

Accuracy is easy to read but **misleading for rare failures**: a model that always says "healthy" is correct for
more than 99.8% of windows and still catches no failures. We therefore report accuracy together with balanced
accuracy, precision, recall, F1 and MCC (Matthews correlation), at the alarm threshold that gives a **1% false-alarm
rate** on healthy windows (seed 0, 30-day horizon).

In [ ]:
def threshold_at_far(y, p, far=0.01):
    neg = np.sort(p[y == 0])
    k = int(np.floor(far * len(neg)))
    return neg[len(neg) - k - 1] + 1e-12

def classification_rows(df, systems, far=0.01):
    y_all = df[f"y_{H}"].to_numpy()
    ok = ~np.isnan(y_all)
    y = y_all[ok].astype(int)
    rows = [{"system": "Always 'healthy' (no model)", "threshold": "–",
             "accuracy": accuracy_score(y, np.zeros_like(y)),
             "balanced acc.": balanced_accuracy_score(y, np.zeros_like(y)),
             "precision": 0.0, "recall": 0.0, "F1": 0.0, "MCC": 0.0,
             "alarms": 0, "true alarms": 0}]
    for s in systems:
        col = f"p{H}__{s}"
        if col not in df:
            continue
        p = df[col].to_numpy()[ok]
        thr = threshold_at_far(y, p, far)
        yhat = (p >= thr).astype(int)
        rows.append({"system": name(s), "threshold": f"{thr:.4f}",
                     "accuracy": accuracy_score(y, yhat),
                     "balanced acc.": balanced_accuracy_score(y, yhat),
                     "precision": precision_score(y, yhat, zero_division=0),
                     "recall": recall_score(y, yhat),
                     "F1": f1_score(y, yhat), "MCC": matthews_corrcoef(y, yhat),
                     "alarms": int(yhat.sum()), "true alarms": int((yhat & y).sum())})
    return pd.DataFrame(rows)

for set_name, df in preds.items():
    t = classification_rows(df, KEY)
    base = np.nanmean(df[f"y_{H}"])
    display(Markdown(f"**{set_name.capitalize()} set** — {int(np.nansum(df[f'y_{H}'])):,} failing windows out of "
                     f"{int((~df[f'y_{H}'].isna()).sum()):,} ({100 * base:.3f}%)"))
    display(t.style.format({c: "{:.4f}" for c in ["accuracy", "balanced acc.", "precision", "recall",
                                                   "F1", "MCC"]}).hide(axis="index"))
display(Markdown("*Majority voting outputs only a few distinct scores (shares of agents voting 'fail'), so its "
                 "threshold cannot hit exactly 1% false alarms and it raises far more alarms.*"))

In [ ]:
df = preds["test"]
ok = ~df[f"y_{H}"].isna()
y = df.loc[ok, f"y_{H}"].astype(int).to_numpy()
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, s in zip(axes, ["full_system", "lgbm_global"]):
    p = df.loc[ok, f"p{H}__{s}"].to_numpy()
    yhat = (p >= threshold_at_far(y, p)).astype(int)
    cm = confusion_matrix(y, yhat)
    ax.imshow(np.log10(cm + 1), cmap="Blues")
    for (i, j), v in np.ndenumerate(cm):
        ax.text(j, i, f"{v:,}", ha="center", va="center", fontsize=11,
                color="white" if np.log10(v + 1) > 3 else "black")
    ax.set_xticks([0, 1], ["predicted healthy", "predicted failing"])
    ax.set_yticks([0, 1], ["actually healthy", "actually failing"])
    ax.set_title(f"{name(s)} (test, 1% FAR)")
    ax.grid(False)
plt.tight_layout(); plt.show()

**Decisions, not just alarms.** The system recommends an action (monitor / schedule replacement / replace now,
or escalate to a human). With hindsight, the best action for each window is known from its labels
(replace now if it failed within 7 days, schedule if within 30 days, otherwise monitor). The table compares the
system's cost-based action with that hindsight-best action.

In [ ]:
def hindsight_action(df):
    y7, y30 = df["y_7"].to_numpy(), df[f"y_{H}"].to_numpy()
    out = np.where(y30 == 1, "schedule_replacement", "monitor").astype(object)
    out[y7 == 1] = "replace_now"
    out[np.isnan(y30)] = None
    return out

for set_name, df in preds.items():
    if "base_action__full_system" not in df:
        continue
    best = hindsight_action(df)
    ok = pd.notna(best)
    tab = pd.crosstab(pd.Series(best[ok], name="hindsight-best action"),
                      pd.Series(df.loc[ok, "base_action__full_system"].to_numpy(),
                                name="system's cost-based action"))
    acc = (df.loc[ok, "base_action__full_system"].to_numpy() == best[ok]).mean()
    acc_monitor = (best[ok] == "monitor").mean()
    a = agg[agg["set"] == set_name].set_index("system")
    display(Markdown(
        f"**{set_name.capitalize()} set** (cost ratio R = {R:g}) — the system's action equals the hindsight-best "
        f"action for {acc:.2%} of windows, and {(df['action__full_system'] == 'escalate_to_human').mean():.2%} "
        f"of windows are escalated to a human. Simply always monitoring would 'agree' with hindsight for "
        f"{acc_monitor:.2%} of windows — but it misses every failure, so its cost is higher: "
        f"{a.loc['full_system', f'cost_always_monitor_r{R:g}_mean']:.4f} per window vs "
        f"{a.loc['full_system', f'cost_model_r{R:g}_mean']:.4f} for the system. Again, cost is the fair measure."))
    display(tab)

## 4. Result graphs

In [ ]:
CURVE_SYSTEMS = [s for s in KEY if f"p{H}__{s}" in preds["test"]]
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
for col, (set_name, df) in enumerate(preds.items()):
    ok = ~df[f"y_{H}"].isna()
    y = df.loc[ok, f"y_{H}"].to_numpy()
    for s in CURVE_SYSTEMS:
        p = df.loc[ok, f"p{H}__{s}"].to_numpy()
        fpr, tpr, _ = roc_curve(y, p)
        axes[0, col].plot(fpr, tpr, lw=1.5, label=f"{name(s)} (AUC {roc_auc_score(y, p):.3f})")
        prec, rec, _ = precision_recall_curve(y, p)
        axes[1, col].plot(rec, prec, lw=1.5, label=f"{name(s)} (AP {average_precision_score(y, p):.3f})")
    axes[0, col].plot([0, 1], [0, 1], "k:", lw=0.8)
    axes[0, col].set(title=f"ROC curve, {H}-day ({set_name}, seed {SEEDS[0]})",
                     xlabel="false-alarm rate", ylabel="recall (true-positive rate)")
    axes[1, col].axhline(y.mean(), color="k", ls=":", lw=0.8, label=f"random ({y.mean():.4f})")
    axes[1, col].set(title=f"Precision–recall, {H}-day ({set_name})", xlabel="recall",
                     ylabel="precision", yscale="log", ylim=(max(y.mean() / 3, 1e-4), 1))
    for r in (0, 1):
        axes[r, col].legend(fontsize=7)
plt.tight_layout(); plt.show()

In [ ]:
def reliability(y, p, bins=10):
    order = np.argsort(p)
    return pd.DataFrame([{"pred": p[i].mean(), "obs": y[i].mean()}
                         for i in np.array_split(order, bins) if len(i)])

def ece(y, p, bins=10):
    r = reliability(y, p, bins)
    return float(np.mean(np.abs(r["pred"] - r["obs"])))  # equal-size bins: equal weights

display(Markdown("**Calibration.** Windows are sorted by predicted probability and split into 10 equal-size "
                 "groups; each point compares the average prediction with the real failure rate in that group. "
                 "Points on the dashed line mean the probabilities can be trusted (log scale; groups with no "
                 "failures are drawn at the bottom edge)."))
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
for ax, (set_name, df) in zip(axes, preds.items()):
    ok = ~df[f"y_{H}"].isna()
    y = df.loc[ok, f"y_{H}"].to_numpy()
    floor = 1e-5
    for s in ["full_system", "no_calibration", "majority_vote"]:
        if f"p{H}__{s}" not in df:
            continue
        p = df.loc[ok, f"p{H}__{s}"].to_numpy()
        r = reliability(y, p)
        ax.plot(r["pred"].clip(lower=floor), r["obs"].clip(lower=floor), "o-", ms=4,
                label=f"{name(s)} (ECE {ece(y, p):.4f})")
    ax.plot([floor, 1], [floor, 1], "k--", lw=0.8, label="perfect calibration")
    ax.set(xscale="log", yscale="log", xlim=(floor, 1), ylim=(floor * 0.8, 1),
           title=f"Calibration, {H}-day ({set_name})", xlabel="mean predicted probability",
           ylabel="observed failure rate")
    ax.legend(fontsize=7, loc="upper left")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for ax, set_name in zip(axes, ("test", "unseen")):
    path = FINAL / "tables" / f"cost_{set_name}.csv"
    if not path.exists():
        continue
    c = pd.read_csv(path)
    for s in KEY:
        d = c[c["system"] == s]
        ax.plot(d["cost_ratio"], d["cost (escalation: model)"], "o-", label=name(s))
    d = c[c["system"] == "full_system"]
    ax.plot(d["cost_ratio"], d["always monitor"], "k:", label="always monitor")
    ax.plot(d["cost_ratio"], d["hindsight optimum"], "k--", label="hindsight optimum")
    ax.set(xscale="log", yscale="log", title=f"Realised cost vs cost ratio ({set_name})",
           xlabel="cost ratio R (missed failure / planned replacement)",
           ylabel="mean cost per window (lower is better)")
axes[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

path = FINAL / "tables" / "selective.csv"
if path.exists():
    sel = pd.read_csv(path)
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
    for ax, set_name in zip(axes, ("test", "unseen")):
        for s in KEY:
            d = sel[(sel["set"] == set_name) & (sel["system"] == s)]
            if len(d):
                ax.plot(d["coverage"], d["selective_accuracy"], "o-", label=name(s))
        ax.set(title=f"Decision accuracy vs coverage ({set_name})",
               xlabel="share of cases decided automatically (rest → human)",
               ylabel="decisions equal to hindsight-best")
    axes[1].legend(fontsize=7)
    plt.tight_layout(); plt.show()

## 5. Ablation study — remove one component at a time

Each row is the full system with **one component removed or replaced**; everything else is unchanged (same data,
splits, seeds and trained models — removing an agent refits only the fusion, on validation data).
Δ = ablation − full system (mean over seeds). The p-value is a paired, drive-clustered bootstrap on AUPRC
(seed 0, 300 resamples): small p means the change is unlikely to be chance.

In [ ]:
ABLATIONS = ["no_media_health", "no_mechanical", "no_thermal", "no_usage_wear", "no_case_retrieval",
             "no_knowledge_retrieval", "no_calibration", "majority_vote", "no_cost_policy",
             "no_drift_recalibration"]

def weighted_ap(y_sorted, w_sorted):
    tp = np.cumsum(w_sorted * y_sorted)
    fp = np.cumsum(w_sorted * (1 - y_sorted))
    return float(np.sum(tp / np.maximum(tp + fp, 1e-12) * w_sorted * y_sorted) / tp[-1])

N_BOOT = 300

def fmt_p(p):
    return "–" if pd.isna(p) else f"< {1 / N_BOOT:.3f}" if p == 0 else f"{p:.3f}"

def bootstrap_pvalues(df, systems, ref="full_system", n_boot=N_BOOT, seed=0):
    ok = ~df[f"y_{H}"].isna()
    y = df.loc[ok, f"y_{H}"].to_numpy()
    codes = pd.factorize(df.loc[ok, "drive"])[0]
    rng = np.random.default_rng(seed)
    g = codes.max() + 1
    weights = [np.bincount(rng.integers(0, g, g), minlength=g)[codes].astype(float)
               for _ in range(n_boot)]
    def boot(s):
        order = np.argsort(-df.loc[ok, f"p{H}__{s}"].to_numpy(), kind="stable")
        ys = y[order]
        return np.array([weighted_ap(ys, w[order]) for w in weights])
    ref_b = boot(ref)
    out = {}
    for s in systems:
        if f"p{H}__{s}" in df:
            d = boot(s) - ref_b
            out[s] = min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))
    return out

def ablation_table(set_name):
    a = agg[agg["set"] == set_name].set_index("system")
    full = a.loc["full_system"]
    pv = bootstrap_pvalues(preds[set_name], ABLATIONS) if set_name in preds else {}
    rows = []
    for s in ABLATIONS:
        if s not in a.index:
            continue
        r = a.loc[s]
        rows.append({"removed / changed": name(s),
                     f"AUPRC {H}d": r[f"auprc_{H}_mean"],
                     "Δ AUPRC": r[f"auprc_{H}_mean"] - full[f"auprc_{H}_mean"],
                     "Δ AUPRC (%)": 100 * (r[f"auprc_{H}_mean"] / full[f"auprc_{H}_mean"] - 1),
                     "Δ ROC-AUC": r[f"roc_auc_{H}_mean"] - full[f"roc_auc_{H}_mean"],
                     "Δ ECE": r[f"ece_{H}_mean"] - full[f"ece_{H}_mean"],
                     f"Δ cost (R={R:g})": r[f"cost_model_r{R:g}_mean"] - full[f"cost_model_r{R:g}_mean"],
                     "p-value (AUPRC)": pv.get(s, np.nan),
                     "what it does": metrics["descriptions"].get(s, "")})
    return pd.DataFrame(rows), full

ABL = {}
for set_name in ("test", "unseen"):
    t, full = ablation_table(set_name)
    ABL[set_name] = t
    display(Markdown(f"**{set_name.capitalize()} set** — full system: AUPRC {full[f'auprc_{H}_mean']:.4f}, "
                     f"ROC-AUC {full[f'roc_auc_{H}_mean']:.4f}, cost {full[f'cost_model_r{R:g}_mean']:.4f}"))
    display(t.style.format({f"AUPRC {H}d": "{:.4f}", "Δ AUPRC": "{:+.4f}", "Δ AUPRC (%)": "{:+.1f}",
                            "Δ ROC-AUC": "{:+.4f}", "Δ ECE": "{:+.4f}", f"Δ cost (R={R:g})": "{:+.4f}",
                            "p-value (AUPRC)": fmt_p}).hide(axis="index"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
t = ABL["test"].sort_values("Δ AUPRC")
HELPS, HURTS = "#2563eb", "#f59e0b"
axes[0].barh(t["removed / changed"], t["Δ AUPRC"],
             color=[HELPS if v < 0 else HURTS for v in t["Δ AUPRC"]])
axes[0].axvline(0, color="k", lw=0.8)
axes[0].set(title=f"Change in AUPRC ({H}-day, test) when a component is removed",
            xlabel="Δ AUPRC  (blue = removing it hurts, so the component helps)")
c = ABL["test"].sort_values(f"Δ cost (R={R:g})", ascending=False)
axes[1].barh(c["removed / changed"], c[f"Δ cost (R={R:g})"],
             color=[HELPS if v > 0 else HURTS for v in c[f"Δ cost (R={R:g})"]])
axes[1].axvline(0, color="k", lw=0.8)
axes[1].set(title=f"Change in maintenance cost (R={R:g}, test)",
            xlabel="Δ cost per window  (blue = removing it costs more, so the component saves money)")
plt.tight_layout(); plt.show()

In [ ]:
def line(t, s):
    r = t.loc[name(s)]
    label = name(s)[2:] if name(s).startswith("– ") else name(s)
    return (f"{label}: AUPRC {r['Δ AUPRC (%)']:+.1f}%, cost {r[f'Δ cost (R={R:g})']:+.4f}, "
            f"p {fmt_p(r['p-value (AUPRC)'])}")

for set_name in ("test", "unseen"):
    t = ABL[set_name].set_index("removed / changed")
    display(Markdown(f"**Ablation, {set_name} set** (Δ = without the component − full system):\n\n" + "\n".join(
        f"- {line(t, s)}" for s in ABLATIONS if name(s) in t.index)))

display(Markdown('''
**How to read it**
- **Media-health agent** (bad-sector counters) is the single most important component on both sets.
- **Calibration** and **calibrated fusion** (vs. majority voting) matter most for **cost**: without them the
  probabilities are wrong, so the cost-based decisions are wrong. On the unseen drive model the uncalibrated
  scores *rank* better (higher AUPRC, though not significantly), yet they still cost more — ranking and
  decisions are different things, and calibration fitted on the training models transfers imperfectly.
- **Case retrieval** gives a small, significant gain on the later period but not on the unseen model.
- **Knowledge retrieval** and the **planner** never change the numbers (they only ground and organise the
  explanation); their effect is measured in Section 6.
- **Drift recalibration**, **thermal** and **usage/wear** agents change AUPRC very little. A tiny p-value with a
  tiny Δ (e.g. drift recalibration) means the change is consistent but too small to matter.
'''))

## 6. LLM ablation — live graph runs

The probabilities come from trained, calibrated models, so the LLM parts (planner, knowledge retrieval, explainer)
do not change the numbers above. Their effect is measured by running the **real LangGraph pipeline with the live
Qwen3.5-4B server** on a stratified sample of test windows, once per variant:

| Variant | What changes |
|---|---|
| Full system | LLM planner + 7 agents + case and knowledge retrieval + LLM explainer |
| – LLM planner | every agent runs in a fixed order (no LLM planning) |
| – knowledge retrieval | no cited documentation snippets for the explainer |
| Agents without retrieval | no case memory and no knowledge snippets |
| Single LLM agent | one agent over all features instead of seven specialists |

In [ ]:
llm_metrics = LLM / "metrics.json"
lm = json.loads(llm_metrics.read_text(encoding="utf-8")).get("llm_sample") if llm_metrics.exists() else None
if lm is None:
    display(Markdown("> ⚠️ No live-LLM ablation in this bundle. Run "
                     "`configs/experiments/ablation_llm.yaml` with the Colab server, then re-export."))
else:
    tab = pd.DataFrame(lm["table"])
    live = lm["mode"] == "live" and tab["llm_available_share"].min() >= 0.95
    if not live:
        display(Markdown(f"> ⚠️ **Not a live-LLM result** (mode: {lm['mode']}, minimum LLM availability "
                         f"{tab['llm_available_share'].min():.0%}). {lm.get('note', '')}"))
    display(Markdown(f"Sample: **{lm['n']} test windows** ({lm['positives']} failing within {H} days), "
                     f"LLM mode **{lm['mode']}**."))
    cols = {"system": "variant", "llm_available_share": "LLM available", "mean_llm_calls": "LLM calls / run",
            "mean_agents_run": "agents run", "planner_llm_share": "plan from LLM",
            "llm_explanation_share": "explanation by LLM",
            "explanations_with_valid_citation": "valid citations", "mean_citations": "citations / run",
            "numbers_grounded_share": "numbers grounded", "agent_failed_share": "runs with failed agent",
            "replanned_share": "re-planned", "escalation_share": "escalated",
            "mean_latency_s": "latency mean (s)", "p95_latency_s": "latency p95 (s)",
            f"auprc_{H}_sample": f"AUPRC {H}d (sample)"}
    show = tab[[c for c in cols if c in tab]].rename(columns=cols)
    show["variant"] = show["variant"].map(name)
    display(show.style.format(precision=3).hide(axis="index"))

    fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
    labels = [name(s) for s in tab["system"]]
    for ax, (c, title) in zip(axes, [(f"auprc_{H}_sample", f"AUPRC {H}-day on the sample (higher is better)"),
                                     ("mean_citations", "Evidence citations per explanation"),
                                     ("mean_latency_s", "Mean latency per analysis (s)")]):
        if c in tab:
            ax.barh(labels, tab[c], color="#7c3aed")
            ax.set_title(title, fontsize=10)
    plt.tight_layout(); plt.show()

    v = tab.set_index("system")
    def get(s, c):
        return v.loc[s, c] if s in v.index and c in v else np.nan
    f, npl, nk, nr, sa = "full_system", "no_planner", "no_knowledge_retrieval", "multi_agent_no_rag", "single_llm_agent"
    display(Markdown(f'''
**Reading the LLM ablation** (AUPRC here is on the stratified sample, which is
{100 * lm["positives"] / lm["n"]:.0f}% failures, so it is much higher than on the real test set and is only for
comparing variants with each other):
- **Planner:** with it, the LLM chose on average {get(f, "mean_agents_run"):.2f} of 7 agents per drive instead of
  running all of them, at a cost of {get(f, "mean_latency_s") - get(npl, "mean_latency_s"):.0f} s extra latency;
  sample AUPRC {get(f, f"auprc_{H}_sample"):.3f} with it vs {get(npl, f"auprc_{H}_sample"):.3f} without.
- **Knowledge retrieval:** explanations cite {get(f, "mean_citations"):.1f} evidence items with it vs
  {get(nk, "mean_citations"):.1f} without; removing all retrieval drops this to {get(nr, "mean_citations"):.1f}.
- **Seven specialists vs one agent:** sample AUPRC {get(f, f"auprc_{H}_sample"):.3f} vs {get(sa, f"auprc_{H}_sample"):.3f};
  the single agent is faster ({get(sa, "mean_latency_s"):.0f} s vs {get(f, "mean_latency_s"):.0f} s) but escalates
  far fewer cases ({100 * get(sa, "escalation_share"):.0f}% vs {100 * get(f, "escalation_share"):.0f}%) because there
  is no agent disagreement to detect.
- **Faithfulness:** in every variant {100 * tab["numbers_grounded_share"].min():.0f}% of numbers in the explanations
  matched the evidence and {100 * tab["explanations_with_valid_citation"].min():.0f}% of explanations had valid
  citations; LLM availability was {100 * tab["llm_available_share"].min():.0f}% in every run.
'''))

## 7. Comparison with existing work

### 7.1 What each approach provides
✓ = an explicit part of the published method · ◐ = partly · – = not a focus of that work (based on our reading of
the papers; see the references in the project report). The published numbers are **not** copied here, because
those studies used different data, years, drive models and metrics, so they cannot be compared directly.

| Approach | Failure prediction from SMART | Calibrated probabilities | Several horizons (7/14/30 d) | Cost-based action | Abstain / escalate to human | Test on an unseen drive model | Retrieval of similar past drives | Cited natural-language explanation | LLM kept out of the numbers |
|---|---|---|---|---|---|---|---|---|---|
| Murray et al., 2005 (SVM, multiple-instance naive Bayes) | ✓ | – | – | – | – | – | – | – | n/a |
| Pinheiro et al., 2007 (Google fleet study) | analysis only | – | – | – | – | – | – | – | n/a |
| Botezatu et al., 2016 (change-point features, regularised classifiers) | ✓ | – | – | – | – | ◐ (transfer between models) | – | – | n/a |
| Lu et al., 2020 (SMART + performance + location, LSTM) | ✓ | – | – | – | – | – | – | – | n/a |
| Typical Backblaze ML project (one GBDT / random forest on all SMART) | ✓ | – | – | – | – | – | – | – | n/a |
| General LLM-agent frameworks (ReAct, AutoGen) | – | – | – | – | – | – | ◐ (document retrieval) | ◐ (reasoning traces) | – (LLM reasons over numbers) |
| **This project** | ✓ | ✓ | ✓ | ✓ | ✓ | ✓ | ✓ | ✓ | ✓ |

### 7.2 Same data, same splits: our system against stand-ins for existing approaches
To compare fairly, each kind of existing approach is represented by a system **trained and tested on exactly our
data and splits**: one gradient-boosted model on all SMART features (the most common Backblaze approach), a deep
sequence model over the raw 30-day series (as in recurrent/convolutional papers), and simple voting of
per-family models.

In [ ]:
STANDINS = [("lgbm_global", "one GBDT on all SMART attributes (typical Backblaze project)"),
            ("sequence", "deep sequence model over the raw 30-day series (LSTM/CNN papers)"),
            ("majority_vote", "simple voting of per-family models"),
            ("multi_agent_no_rag", "our specialist agents without retrieval"),
            ("full_system", "our full system")]
rows = []
for set_name in ("test", "unseen"):
    a = agg[agg["set"] == set_name].set_index("system")
    full = a.loc["full_system"]
    for s, desc in STANDINS:
        if s not in a.index:
            continue
        r = a.loc[s]
        rows.append({"set": set_name, "system": name(s), "represents": desc,
                     f"AUPRC {H}d": r[f"auprc_{H}_mean"], f"ROC-AUC {H}d": r[f"roc_auc_{H}_mean"],
                     "Recall @1% FAR": r[f"recall@far0.01_{H}_mean"], f"ECE {H}d": r[f"ece_{H}_mean"],
                     "Detected (%)": 100 * r["lead_detected_share_mean"],
                     f"Cost R={R:g}": r[f"cost_model_r{R:g}_mean"],
                     "ours ÷ this (AUPRC)": full[f"auprc_{H}_mean"] / r[f"auprc_{H}_mean"]})
cmp = pd.DataFrame(rows)
display(cmp.style.format({f"AUPRC {H}d": "{:.4f}", f"ROC-AUC {H}d": "{:.4f}", "Recall @1% FAR": "{:.3f}",
                          f"ECE {H}d": "{:.4f}", "Detected (%)": "{:.1f}", f"Cost R={R:g}": "{:.4f}",
                          "ours ÷ this (AUPRC)": "{:.2f}×"}).hide(axis="index"))

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
t = cmp[cmp["set"] == "test"]
for ax, (c, title) in zip(axes, [(f"AUPRC {H}d", f"AUPRC {H}-day (higher is better)"),
                                 (f"ROC-AUC {H}d", f"ROC-AUC {H}-day (higher is better)"),
                                 (f"Cost R={R:g}", f"Cost per window, R={R:g} (lower is better)")]):
    ax.barh(t["system"], t[c], color=["#2563eb" if "ours" in s.lower() else "#94a3b8" for s in t["system"]])
    ax.set_title(title, fontsize=10)
plt.tight_layout(); plt.show()

## 8. Conclusions

In [ ]:
a = agg[agg["set"] == "test"].set_index("system")
u = agg[agg["set"] == "unseen"].set_index("system")
f, g, s, m, nc, mv = (a.loc[k] for k in ["full_system", "lgbm_global", "sequence", "multi_agent_no_rag",
                                         "no_calibration", "majority_vote"])
abl = ABL["test"].set_index("removed / changed")
pv_case = abl.loc[name("no_case_retrieval"), "p-value (AUPRC)"] if name("no_case_retrieval") in abl.index else np.nan
drift_d = abs(a.loc["no_drift_recalibration", f"auprc_{H}_mean"] - f[f"auprc_{H}_mean"])
display(Markdown(f'''
**H1 — specialist agents beat one model.** Full system AUPRC {f[f"auprc_{H}_mean"]:.3f} vs
{g[f"auprc_{H}_mean"]:.3f} for one LightGBM ({f[f"auprc_{H}_mean"] / g[f"auprc_{H}_mean"]:.1f}×) and
{s[f"auprc_{H}_mean"]:.3f} for the 1D-CNN ({f[f"auprc_{H}_mean"] / s[f"auprc_{H}_mean"]:.1f}×);
ROC-AUC {f[f"roc_auc_{H}_mean"]:.3f}; {100 * f["lead_detected_share_mean"]:.0f}% of failing drives detected
about {f["lead_mean_lead_days_mean"]:.0f} days early. The media-health agent is the most important component.

**H2 — retrieval of similar past drives helps a little.** AUPRC {m[f"auprc_{H}_mean"]:.3f} → {f[f"auprc_{H}_mean"]:.3f}
on the later period (p = {pv_case:.3f} vs no case retrieval); on the unseen model
{u.loc["multi_agent_no_rag", f"auprc_{H}_mean"]:.3f} → {u.loc["full_system", f"auprc_{H}_mean"]:.3f}.

**H3 — calibration and the cost policy save money.** Cost per window at R = {R:g}: {f[f"cost_model_r{R:g}_mean"]:.3f}
(full) vs {nc[f"cost_model_r{R:g}_mean"]:.3f} without calibration and {mv[f"cost_model_r{R:g}_mean"]:.3f} with
majority voting; ECE {f[f"ece_{H}_mean"]:.4f} vs {nc[f"ece_{H}_mean"]:.4f}.

**H4 — drift recalibration.** It changed AUPRC by only {drift_d:.4f}, so it is not supported on this data.

**Limitations.** Failures are only about {100 * np.nanmean(preds["test"][f"y_{H}"]):.2f}% of windows, so precision at a
1% false-alarm rate stays low (Section 3); the cost ratio R is an assumption (Section 4 shows the sweep); some
probabilities are extreme; the data covers two years and five drive models.
'''))